# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanmustafa119/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import duckdb
import pandas as pd
import numpy as np

print("Imports successful.")

Imports successful.


In [2]:
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

login(token=HF_TOKEN)

print("✅ Hugging Face authentication successful")

Enter your Hugging Face token: ··········
✅ Hugging Face authentication successful


In [3]:
con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute("""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("✅ DuckDB is connected to Hugging Face")

✅ DuckDB is connected to Hugging Face


In [4]:
test_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

test_df = con.sql(test_query).df()

print("✅ Dataset access successful")
print("Rows returned:", len(test_df))

display(test_df)

✅ Dataset access successful
Rows returned: 5


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print("Number of columns:", len(test_df.columns))

for col in test_df.columns:
    print(col)

Number of columns: 31
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


### 1. Method choice and why

My lane is Refresh / Content Opportunity Scoring, where the practical question is which pages should be reviewed first. I will start with Logistic Regression because it is simple and provides an interpretable probability score, then compare it with Random Forest because it can capture nonlinear relationships between search visibility, clicks, CTR, position, and other observed signals.

I will use the predicted probability as the ranking score and evaluate the ranking with Precision@50. This fits the decision-support goal better than relying only on classification accuracy.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Split design

I will use a client-holdout split rather than randomly splitting individual content rows. Approximately 20% of clients will be held out completely from training, so pages from the same client cannot appear in both train and test.

This is a more honest test for the intended use because the model is evaluated on content from clients it did not see during training. I will use a fixed random seed of 42 so the split is reproducible.


In [7]:
model_query = """
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS march_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,

    COALESCE(a.april_impressions, 0) AS april_impressions,
    COALESCE(a.april_clicks, 0) AS april_clicks

FROM march m
LEFT JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id

WHERE m.march_impressions >= 100
"""

model_df = con.sql(model_query).df()

print("Modeling rows:", len(model_df))

display(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 101441


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,april_impressions,april_clicks
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,345.0,1.0,38.879856,187.0,0.0
1,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,854.0,1.0,8.792961,202.0,1.0
2,client_62f4a7e64f5e0096,content_3f87f49c36774e23,205.0,0.0,34.603295,93.0,0.0
3,client_62f4a7e64f5e0096,content_64706c8afebebb8c,666.0,2.0,5.928198,131.0,1.0
4,client_62f4a7e64f5e0096,content_8809e466350d5b98,174.0,0.0,6.880781,113.0,0.0


In [8]:
model_df["impression_change_pct"] = (
    (model_df["april_impressions"] - model_df["march_impressions"])
    / model_df["march_impressions"]
) * 100

model_df["decline_label"] = (
    model_df["impression_change_pct"] <= -20
).astype(int)

print(
    "Declining pages:",
    model_df["decline_label"].sum()
)

print(
    "Decline rate:",
    round(model_df["decline_label"].mean() * 100, 2),
    "%"
)

display(
    model_df[
        [
            "march_impressions",
            "april_impressions",
            "impression_change_pct",
            "decline_label"
        ]
    ].head(10)
)

Declining pages: 52533
Decline rate: 51.79 %


,march_impressions,april_impressions,impression_change_pct,decline_label
0,345.0,187.0,-45.797101,1
1,854.0,202.0,-76.346604,1
2,205.0,93.0,-54.634146,1
3,666.0,131.0,-80.330330,1
4,174.0,113.0,-35.057471,1
5,561.0,137.0,-75.579323,1
6,103.0,66.0,-35.922330,1
7,351.0,99.0,-71.794872,1
8,4815.0,1059.0,-78.006231,1
9,884.0,367.0,-58.484163,1


In [9]:
model_df["march_ctr"] = (
    model_df["march_clicks"]
    / model_df["march_impressions"]
) * 100

model_df["log_march_impressions"] = np.log1p(
    model_df["march_impressions"]
)

model_df["log_march_clicks"] = np.log1p(
    model_df["march_clicks"]
)

features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "log_march_impressions",
    "log_march_clicks"
]

model_df[features + ["decline_label"]].describe()

,march_impressions,march_clicks,march_ctr,march_avg_position,log_march_impressions,log_march_clicks,decline_label
count,101441.000000,101441.000000,101441.000000,101441.000000,101441.000000,101441.000000,101441.000000
mean,2748.357006,8.037924,0.261570,14.764505,6.818532,1.141757,0.517868
std,6945.163794,34.887274,0.419383,14.458647,1.411887,1.229285,0.499683
min,100.000000,0.000000,0.000000,0.101639,4.615121,0.000000,0.000000
25%,283.000000,0.000000,0.000000,5.121419,5.648974,0.000000,0.000000
50%,786.000000,1.000000,0.123762,8.959310,6.668228,0.693147,1.000000
75%,2514.000000,6.000000,0.366972,19.698450,7.830028,1.945910,1.000000
max,617124.000000,5668.000000,15.584416,113.390250,13.332827,8.642768,1.000000


In [10]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["decline_label"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Training clients:",
    train_df["client_hash_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_hash_id"].nunique()
)

print(
    "Shared clients:",
    len(
        set(train_df["client_hash_id"])
        &
        set(test_df["client_hash_id"])
    )
)

Training rows: 94403
Test rows: 7038
Training clients: 35
Test clients: 9
Shared clients: 0


In [11]:
print(
    "Train decline rate:",
    round(train_df["decline_label"].mean() * 100, 2),
    "%"
)

print(
    "Test decline rate:",
    round(test_df["decline_label"].mean() * 100, 2),
    "%"
)

Train decline rate: 51.75 %
Test decline rate: 52.3 %


In [12]:
def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.nlargest(k, "score")

    return top_k["y_true"].mean()


test_df["baseline_score"] = (
    np.log1p(test_df["march_impressions"])
    * (1 - test_df["march_ctr"] / 100)
)

In [13]:
baseline_p50 = precision_at_k(
    test_df["decline_label"],
    test_df["baseline_score"],
    k=50
)

print(
    f"Baseline Precision@50: {baseline_p50:.3f}"
)

Baseline Precision@50: 0.340


In [14]:
print(
    test_df[
        [
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "decline_label",
            "baseline_score"
        ]
    ].head(10)
)

      march_impressions  march_clicks  march_ctr  march_avg_position  \
1163              107.0           0.0   0.000000           18.741472   
1164              122.0           0.0   0.000000           28.376065   
1165              397.0           1.0   0.251889           10.570485   
1166              424.0           0.0   0.000000           34.912309   
1167              223.0           0.0   0.000000            7.816224   
1168              347.0           3.0   0.864553           12.505324   
1169              115.0           1.0   0.869565            8.983163   
1170              720.0           3.0   0.416667            9.067340   
1171              103.0           0.0   0.000000           41.753439   
1172              540.0           1.0   0.185185           13.749226   

      decline_label  baseline_score  
1163              0        4.682131  
1164              0        4.812184  
1165              1        5.971373  
1166              1        6.052089  
1167             

In [15]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "log_march_impressions",
    "log_march_clicks"
]

X_train = train_df[features]
y_train = train_df["decline_label"]

X_test = test_df[features]
y_test = test_df["decline_label"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (94403, 6)
X_test: (7038, 6)


In [16]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

print("✅ Logistic Regression trained")

✅ Logistic Regression trained


In [17]:
logistic_scores = logistic_model.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(logistic_scores))
print("First 10 scores:")
print(logistic_scores[:10])

Number of predictions: 7038
First 10 scores:
[0.55442945 0.53465395 0.5361766  0.56938119 0.61208209 0.40277501
 0.43978714 0.4774412  0.49246547 0.54619265]


In [18]:
logistic_p50 = precision_at_k(
    y_test,
    logistic_scores,
    k=50
)

print(
    f"Logistic Regression Precision@50: {logistic_p50:.3f}"
)

Logistic Regression Precision@50: 0.520


In [19]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

random_forest.fit(X_train, y_train)

print("✅ Random Forest trained")

✅ Random Forest trained


In [20]:
rf_scores = random_forest.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(rf_scores))
print("First 10 scores:")
print(rf_scores[:10])

Number of predictions: 7038
First 10 scores:
[0.53065783 0.48662598 0.46697712 0.48492433 0.6056614  0.33596086
 0.52690821 0.44740508 0.45505121 0.53272787]


In [21]:
rf_p50 = precision_at_k(
    y_test,
    rf_scores,
    k=50
)

print(
    f"Random Forest Precision@50: {rf_p50:.3f}"
)

Random Forest Precision@50: 0.660


In [22]:
results = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_p50,
        logistic_p50,
        rf_p50
    ]
})

results["precision_at_50"] = results["precision_at_50"].round(3)

display(results)

,method,precision_at_50
0,Week-4 baseline,0.34
1,Logistic Regression,0.52
2,Random Forest,0.66


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### 3. Train + compare vs my baseline

I evaluated the baseline and both models on the same client-held-out test set using Precision@50. The Week-4 baseline achieved 0.340 Precision@50, Logistic Regression achieved 0.520, and Random Forest achieved 0.660.

Random Forest performed best on this test set. Its Precision@50 was 0.320 higher than the baseline, meaning 33 of its top 50 ranked pages were labeled as declining compared with 17 of the baseline's top 50.

These are observed results on the held-out clients, so the improvement should be treated as measured decision-support evidence rather than a guarantee of future performance.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
test_results = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "april_impressions",
        "impression_change_pct",
        "decline_label"
    ]
].copy()

test_results["rf_score"] = rf_scores

top_50 = test_results.nlargest(
    50,
    "rf_score"
).copy()

top_50["correct"] = (
    top_50["decline_label"] == 1
)

print("Top 50 predicted declining:")
print("Correct:", top_50["correct"].sum())
print("Incorrect:", (~top_50["correct"]).sum())

display(
    top_50[
        [
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "april_impressions",
            "impression_change_pct",
            "decline_label",
            "rf_score",
            "correct"
        ]
    ].head(20)
)

Top 50 predicted declining:
Correct: 33
Incorrect: 17


,march_impressions,march_clicks,march_ctr,march_avg_position,april_impressions,impression_change_pct,decline_label,rf_score,correct
100999,188.0,0.0,0.000000,87.124803,0.0,-100.000000,1,0.797214,True
1888,192.0,0.0,0.000000,83.362012,136.0,-29.166667,1,0.779725,True
1838,214.0,0.0,0.000000,85.001108,17.0,-92.056075,1,0.776885,True
101272,232.0,0.0,0.000000,90.302463,0.0,-100.000000,1,0.772078,True
62764,13257.0,20.0,0.150864,2.144440,6326.0,-52.281813,1,0.762288,True
11227,361.0,0.0,0.000000,81.547350,16.0,-95.567867,1,0.752164,True
10288,3488.0,3.0,0.086009,1.277769,2374.0,-31.938073,1,0.747089,True
68586,3128.0,5.0,0.159847,1.681366,2945.0,-5.850384,0,0.746588,False
68439,2249.0,2.0,0.088928,0.616099,86.0,-96.176078,1,0.745710,True
101359,182.0,0.0,0.000000,82.407337,0.0,-100.000000,1,0.737213,True


In [24]:
errors = top_50[
    top_50["correct"] == False
].copy()

print("False positives in top 50:", len(errors))

display(
    errors[
        [
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "april_impressions",
            "impression_change_pct",
            "rf_score"
        ]
    ].describe()
)

False positives in top 50: 17


,march_impressions,march_clicks,march_ctr,march_avg_position,april_impressions,impression_change_pct,rf_score
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,2539.823529,0.882353,0.033260,12.427810,4042.470588,75.490204,0.710488
std,4007.002281,1.409005,0.049012,26.809550,6785.577519,147.362269,0.015055
min,116.000000,0.000000,0.000000,1.276815,403.000000,-10.654490,0.695763
25%,1119.000000,0.000000,0.000000,1.914389,1246.000000,3.865979,0.697843
50%,1378.000000,0.000000,0.000000,3.284776,1586.000000,18.650218,0.704141
75%,2152.000000,1.000000,0.074738,3.815851,3253.000000,64.136913,0.719417
max,17617.000000,5.000000,0.159847,87.959570,28916.000000,556.451613,0.746588


In [25]:
rf_estimator = random_forest.named_steps["model"]

importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf_estimator.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

,feature,importance
2,march_ctr,0.311222
3,march_avg_position,0.211529
5,log_march_clicks,0.134927
1,march_clicks,0.123925
4,log_march_impressions,0.110900
0,march_impressions,0.107496


In [26]:
# Feature importance
importance_df = importance_df.copy()

display(importance_df)

# Top-50 error summary
print("Top-50 correct:", top_50["correct"].sum())
print("Top-50 incorrect:", (~top_50["correct"]).sum())

print("\nFalse-positive median values:")
display(
    errors[
        [
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "march_avg_position",
            "impression_change_pct"
        ]
    ].median()
)

,feature,importance
2,march_ctr,0.311222
3,march_avg_position,0.211529
5,log_march_clicks,0.134927
1,march_clicks,0.123925
4,log_march_impressions,0.110900
0,march_impressions,0.107496


Top-50 correct: 33
Top-50 incorrect: 17

False-positive median values:


,0
march_impressions,1378.000000
march_clicks,0.000000
march_ctr,0.000000
march_avg_position,3.284776
impression_change_pct,18.650218


### 4. Errors and interpretation

The Random Forest correctly identified 33 of the top 50 ranked pages as declining, while 17 were false positives. The false positives often had low CTR, but their median April impression change was positive at +18.65%, showing that low CTR alone can make a page look risky even when search visibility is increasing.

The model relied most on March CTR (importance 0.311) and average position (0.212), followed by log March clicks (0.135) and March clicks (0.124). These are measured feature-importance values, not evidence of causation. In practice, the model appears useful for prioritizing pages for review, but the ranked results still need human context before action.


## Self-check

Before you submit, confirm each line honestly:

- ✔️ Every section above is filled — markdown thinking AND the code that backs it
- ✔️ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔️ No client names, URLs, or private queries anywhere
- ✔️ My claims use careful words: observed, measured, directional, decision-support
- ✔️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.